# mobilenetv3_scratch

**Architecture:** `mobilenetv3_large_100.ra_in1k` (timm)  
Generated by `scripts/create_notebook.py` on 2026-11-20.

Trains the model from scratch (`PRETRAINED = False`: random starting weights, nothing downloaded) on `data/<class>/train`, keeps the epoch with the best accuracy on a validation split carved out of train, then evaluates that checkpoint once on `data/<class>/test`.

Every run is saved to `output/mobilenetv3_scratch/<timestamp>/`: `config.json`, `history.csv`, `best.pth`, `training_curves.png`, `test_metrics.json`, `confusion_matrix.png`, `misclassified.png`, `top_confusions.csv`, and `predictions.csv` for the hidden test set.

Runs unchanged on this machine (CPU or GPU), Google Colab and Kaggle; for Colab/Kaggle see README > *Colab and Kaggle*.

To reset this notebook: `python scripts/create_notebook.py mobilenetv3_scratch --arch mobilenetv3_large_100.ra_in1k --lr 1e-3 --epochs 60 --set PRETRAINED=False --set NUM_WORKERS=4 --force`

In [ ]:
# ---- settings for this model (edit freely) ----
MODEL_NAME = 'mobilenetv3_scratch'  # results go to output/<MODEL_NAME>/<timestamp>/
ARCH = 'mobilenetv3_large_100.ra_in1k'  # any timm model name, e.g. timm.list_models('*convnext*', pretrained=True)
PRETRAINED = False              # True = start from ImageNet weights; False = train from scratch
DEVICE = 'auto'                 # "auto" = GPU if available, otherwise CPU; or force "cuda" / "cpu"
IMG_SIZE = 224                  # input resolution in pixels
EPOCHS = 60
BATCH_SIZE = 32                 # lower this if the GPU runs out of memory
LR = 1e-3                       # peak learning rate (AdamW, cosine schedule, 1 warm-up epoch)
WEIGHT_DECAY = 0.05
LABEL_SMOOTHING = 0.1
TRAIN_MODE = 'finetune'         # "finetune" = all layers; "linear_probe" = classifier only (much faster on CPU)
VAL_FRACTION = 0.1              # share of train held out to pick the best epoch; test is only used at the end
SAMPLES_PER_CLASS = 600         # class-balanced samples per epoch (butterfly has ~10x more images); None = every image once
PATIENCE = None                 # stop after this many epochs without val improvement; None = run all epochs
LIMIT_PER_CLASS = None          # e.g. 20 for a quick smoke test
NUM_WORKERS = 4                 # image-loading processes
SEED = 42

## Setup
Uses the GPU when one is available, otherwise the CPU (see `DEVICE`). On Colab or Kaggle it first downloads the project code from GitHub, then finds the data: on Colab `imgcls_data.zip` in the Google Drive folder below (runs are saved there too), on Kaggle the dataset attached with *Add Input* (runs go to `/kaggle/working/output`).

In [ ]:
import subprocess
import sys
from pathlib import Path

# find the project (the folder holding imgcls/); on Colab / Kaggle fetch it from GitHub first
REPO_URL = "https://github.com/M0HiT27/Image-Classification.git"
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "imgcls").is_dir()), None)
if ROOT is None:
    on_colab = Path("/content").is_dir() and not Path("/kaggle").is_dir()  # Kaggle has /content too
    ROOT = Path("/content" if on_colab else "/tmp") / "Image-Classification"
    if (ROOT / ".git").is_dir():
        subprocess.run(["git", "-C", str(ROOT), "pull", "-q", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, str(ROOT)], check=True)
if str(ROOT) in sys.path:
    sys.path.remove(str(ROOT))
sys.path.insert(0, str(ROOT))
# forget an imgcls loaded earlier in this session, so re-running this cell after a pull uses the new code
for name in [m for m in sys.modules if m == "imgcls" or m.startswith("imgcls.")]:
    del sys.modules[name]
try:
    import timm  # installed by requirements.txt locally; preinstalled on Kaggle and Colab
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)

from IPython.display import Image, display
import imgcls

# locally: data/ and output/ in the project; Colab: this Google Drive folder; Kaggle: /kaggle/input
imgcls.setup_environment(colab_drive_dir="/content/drive/MyDrive/imgcls")
imgcls.set_seed(SEED)
device = imgcls.get_device(DEVICE)

## Model
Builds the architecture with random weights (nothing is downloaded) and one output per class.

In [ ]:
classes = imgcls.discover_classes()
model, data_cfg = imgcls.create_model(ARCH, num_classes=len(classes), pretrained=PRETRAINED,
                                      train_mode=TRAIN_MODE)

## Data
Train / validation / test counts per class. Training images get random crops, flips, rotations and colour/lighting changes.

In [ ]:
cfg = imgcls.config_from_globals(globals())
data = imgcls.build_data(cfg, classes, data_cfg)

## Train
Prints one line per epoch and saves `best.pth` whenever validation accuracy improves. Interrupting the cell stops training early but keeps the best checkpoint.

In [ ]:
run_dir = imgcls.train(cfg, model, data, device)
display(Image(filename=str(run_dir / "training_curves.png")))

## Evaluate on the test split
Loads `best.pth` and scores it once on the test images. `top_confusions.csv` and `misclassified.png` are the material for the report's failure cases.

In [ ]:
# to evaluate an earlier run instead: run_dir = imgcls.OUTPUT_DIR / MODEL_NAME / "<timestamp>"
metrics = imgcls.evaluate_run(run_dir, data, device)
display(Image(filename=str(run_dir / "confusion_matrix.png")))
if (run_dir / "misclassified.png").exists():
    display(Image(filename=str(run_dir / "misclassified.png")))

## Predict the hidden test set
Writes `predictions.csv` (filename, predicted category) into the run folder.

In [ ]:
HIDDEN_TEST_DIR = None  # e.g. r"D:\hidden_test_images"
if HIDDEN_TEST_DIR:
    imgcls.predict_folder(run_dir / "best.pth", HIDDEN_TEST_DIR,
                          out_csv=run_dir / "predictions.csv", device=device)